# CardioSentry — synthetic ECG image generation (Colab)

Generates the full synthetic paper-ECG corpus from the **v3** harmonized signals.

| | |
|---|---|
| input | `datasets/harmonized.tar` (v3 signals) + `datasets/cardiosentry_pipeline.tar.gz` (patched code) |
| output | `experiments/synthetic_images_v1/` (corpus archives) and `runs/synthetic_images_v1/` (index, logs) |
| expected | ~37,225 records -> ~129,376 images, 8-12 h on 12 vCPU |

**The GPU is not used.** Nothing in this pipeline touches CUDA; the L4 runtime is
only a way to get 12 vCPUs. Section 0 disables CUDA so TensorFlow (imported by the
kit but never called) does not waste time initialising it.

**Everything is resumable.** Stages 2-4 skip work already on disk, so a Colab
disconnect costs only the in-flight chunk. Re-run the same cells.

---
## How to use
1. Run **section 1 (preflight)** first. It hard-stops on any problem *before* the
   8-hour job starts. Do not skip it.
2. Run section 2 (smoke test, ~2 min) and eyeball the image.
3. Run section 3 (full build).
4. Run section 4 (package to Drive).


## 0. Runtime setup


In [ ]:
import os, sys, subprocess, time, json, importlib, importlib.util
from pathlib import Path

# Nothing here uses the GPU. Hiding it stops TF from initialising CUDA on every
# one of the ~100 kit invocations, which is pure overhead.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

DRIVE   = Path('/content/drive/MyDrive/CardioSentry')
DATASETS= DRIVE/'datasets'
CODES   = DRIVE/'codes'/'synthetic_image_generation'
EXPDIR  = DRIVE/'experiments'/'synthetic_images_v1'
RUNDIR  = DRIVE/'runs'/'synthetic_images_v1'

WORK    = Path('/content/work')          # LOCAL disk - never build on Drive
CODE    = WORK/'code'
DATA    = WORK/'data'/'releases'/'v3'
BUILD   = WORK/'build'

from google.colab import drive
drive.mount('/content/drive')
print('mounted')


In [ ]:
# Build parameters. Change these BEFORE the preflight, not after.
PARAMS = dict(
    ECG_RELEASE     = 'v3',
    ECG_WORKERS     = str(os.cpu_count() or 12),
    ECG_IMAGE_BUILD = str(BUILD),
    ECGKIT_DIR      = str(CODE/'ecg-image-kit'),
    ECGKIT_PY       = sys.executable,
    ECG_ANNOTATIONS = '1',   # '0' saves ~45 GB and ~10% time; drop only if you will
                             # never do segmentation / digitisation work
    # Deliberately NOT set, so the config defaults apply:
    #   ECG_LOWPASS_HZ  -> 0.05-150 Hz (diagnostic band)
    #   ECG_PRINT_HEADER-> off (no ID/Name/Date/Age/Sex)
    #   ECG_ROW_GAP     -> 4.4   ECG_RHYTHM_GAP -> 5.5
    #   ECG_DPI         -> 150
)
os.environ.update(PARAMS)
for k, v in PARAMS.items(): print(f'{k:18s} {v}')


## 1. Preflight

Every check below hard-stops the notebook. The point is to fail in the first two
minutes rather than eight hours in.


In [ ]:
FAIL = []
def check(name, ok, detail=''):
    print(('  OK   ' if ok else '  FAIL ') + name + (f'  -- {detail}' if detail else ''))
    if not ok: FAIL.append(name)

def die():
    """Stop the notebook if anything in this section failed."""
    if FAIL:
        msg = 'PREFLIGHT FAILED:\n  - ' + '\n  - '.join(FAIL)
        FAIL.clear()          # so a re-run after a fix starts clean
        raise SystemExit(msg)
    print('  -> section OK')

def section(title):
    FAIL.clear()              # each section judges only its own checks
    print(title)

section('[1] Drive layout')
for p in (DRIVE, DATASETS):
    check(f'{p} exists', p.is_dir())
for p in (CODES, EXPDIR, RUNDIR):
    p.mkdir(parents=True, exist_ok=True)
    check(f'{p.relative_to(DRIVE)} writable', os.access(p, os.W_OK))
die()


In [ ]:
section('[2] Input archives present in datasets/')
SIG_TAR  = DATASETS/'harmonized.tar'
CODE_TAR = DATASETS/'cardiosentry_pipeline.tar.gz'
check('harmonized.tar', SIG_TAR.is_file(), f'{SIG_TAR.stat().st_size/2**30:.2f} GB' if SIG_TAR.is_file() else 'missing')
check('cardiosentry_pipeline.tar.gz', CODE_TAR.is_file(),
      f'{CODE_TAR.stat().st_size/2**20:.0f} MB' if CODE_TAR.is_file() else 'upload it to datasets/')
die()


In [ ]:
section('[3] Local disk and RAM')
st = os.statvfs('/content')
free_gb = st.f_bavail*st.f_frsize/2**30
need = 4.5 + (131.1 if os.environ['ECG_ANNOTATIONS']=='1' else 86.1)
check(f'free disk {free_gb:.0f} GB >= {need:.0f} GB needed', free_gb >= need,
      'set ECG_ANNOTATIONS=0 to need ~91 GB' if free_gb < need else '')
ram = os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/2**30
check(f'RAM {ram:.0f} GB', ram >= 8)
check(f'vCPUs {os.cpu_count()}', (os.cpu_count() or 0) >= 4)
die()


In [ ]:
section('[4] Extract code + signals to LOCAL disk')
# Reading 74,662 signal files through the Drive FUSE layer would take hours.
# Extract once to /content and never touch Drive again until packaging.
def untar(archive, dest, flag):
    dest.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['tar', flag, str(archive), '-C', str(dest)],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:]); raise SystemExit(f'failed to extract {archive.name}')

CODE = WORK/'code'
if not (CODE/'image_pipeline'/'config.py').exists():
    untar(CODE_TAR, CODE, '-xzf')
if not (DATA/'harmonized'/'harmonized_ecg_metadata.csv').exists():
    untar(SIG_TAR, DATA, '-xf')

PIPE = CODE/'image_pipeline'
CSV  = DATA/'harmonized'/'harmonized_ecg_metadata.csv'
check('image_pipeline/config.py', (PIPE/'config.py').is_file())
check('harmonized_ecg_metadata.csv', CSV.is_file())
sigdir = DATA/'harmonized'/'signals'
n_sig = len(list(sigdir.rglob('*.hea'))) if sigdir.is_dir() else 0
check(f'signal headers found: {n_sig:,}', n_sig > 30000, 'expected ~37,329')
die()


In [ ]:
section('[5] Python dependencies')
import importlib, importlib.util
MODS = ('numpy','scipy','pandas','wfdb','PIL','matplotlib','imgaug','qrcode','imageio','yaml')
missing = [m for m in MODS if not importlib.util.find_spec(m)]
if missing:
    print('  installing:', missing)
    subprocess.run([sys.executable,'-m','pip','-q','install',
                    'wfdb','imgaug','qrcode','imageio','pyyaml'], check=False)
    importlib.invalidate_caches()
for m in MODS:
    check(m, importlib.util.find_spec(m) is not None)
die()


In [ ]:
section('[6] Kit patches — the build is WRONG without these')
sys.path.insert(0, str(PIPE))
os.chdir(PIPE)
for m in list(sys.modules):
    if m in ('config','patch_kit'): del sys.modules[m]
import config as C, patch_kit
print(subprocess.run([sys.executable,'patch_kit.py'],capture_output=True,text=True).stdout)
importlib.reload(patch_kit)
check('all kit patches applied', patch_kit.is_patched(),
      'patch_kit reported FAILED above -- do not proceed')
die()


In [ ]:
section('[7] Effective settings — read these carefully')
importlib.reload(C)
print(f'  band-pass       {C.BANDPASS[0]}-{C.BANDPASS[1]} Hz')
print(f'  printed header  {C.PRINT_HEADER}')
print(f'  annotations     {C.ANNOTATIONS}')
print(f'  DPI             {C.DPI}')
print(f'  row / rhythm gap{C.ROW_GAP} / {C.RHYTHM_GAP}  (x10 mm)')
print(f'  workers         {C.WORKERS}')
print(f'  variants        {C.VARIANTS_PER_SPLIT}')
check('band-pass is the diagnostic 0.05-150 Hz', C.BANDPASS == (0.05, 150.0))
check('printed header is OFF', C.PRINT_HEADER is False,
      'ON would drop ~17,875 age-less records and zero the STEMI+AF+LVH class')
check('4 columns', '4' in C.KIT_RENDER_FLAGS)
die()


## 2. Smoke test (~2 min)

Renders 5 records that have **no recorded age**, including the triple-positive
class. This is the specific case that two separate bugs used to drop silently, so
it is the one worth proving before an 8-hour run.


In [ ]:
import pandas as pd
SMOKE = WORK/'smoke'; SMOKE.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV, low_memory=False)
here = CSV.parent
noage = df[df.age.isna() & (df.validation_status=='VALID')]
picks = [noage[noage.cardiosentry_label==l].iloc[0]
         for l in ('STEMI+AF+LVH','STEMI+LVH','STEMI+AF','AF+LVH','LVH')
         if len(noage[noage.cardiosentry_label==l])]
sub = pd.DataFrame(picks).copy()
for c in ('signal_path','header_path','data_path'):
    sub[c] = sub[c].map(lambda p: str((here/p).resolve()))
sub['split'] = 'test'
sub.to_csv(SMOKE/'smoke.csv', index=False)
print(sub[['global_record_id','source_dataset','cardiosentry_label','age','sex']].to_string(index=False))


In [ ]:
def run(script, *args, env=None, quiet=False):
    """Run a pipeline stage. env defaults to the CURRENT os.environ at call
    time - never a captured dict, or the full build would inherit the smoke
    test's ECG_IMAGE_BUILD and write into the wrong directory."""
    p = subprocess.run([sys.executable, script, *args], cwd=PIPE,
                       env={**os.environ, **(env or {})},
                       capture_output=True, text=True)
    out = p.stdout.strip()
    print(out if not quiet else '\n'.join(out.splitlines()[-6:]))
    if p.returncode:
        print(p.stderr[-2000:]); raise SystemExit(f'{script} failed')
    return out

SMOKE_ENV = {'ECG_IMAGE_BUILD': str(SMOKE/'build')}
W = os.environ['ECG_WORKERS']
run('stage1_manifest_from_csv.py', '--input-csv', str(SMOKE/'smoke.csv'), env=SMOKE_ENV)
run('stage2_transcode.py', env=SMOKE_ENV)
run('stage3_render.py', '-j', W, env=SMOKE_ENV)
run('stage4_augment.py', '-j', W, env=SMOKE_ENV)


In [ ]:
section('[smoke] verify')
imgs = sorted((SMOKE/'build'/'images').rglob('*.jpg'))
check(f'5 images produced (got {len(imgs)})', len(imgs) == 5,
      'fewer means records are still being dropped')

import numpy as np
from PIL import Image as PILImage
ppmm = C.DPI/25.4
pngs = sorted((SMOKE/'build'/'rendered').rglob('*.png'))
check('clean renders exist', len(pngs) > 0)
if pngs:
    g = np.asarray(PILImage.open(pngs[0]).convert('L'), float)
    band = g[int(14*ppmm):int(30*ppmm), int(15*ppmm):int(95*ppmm)]
    check('no header text on the sheet', band.min() >= 95,
          f'darkest pixel {band.min():.0f}; text would be <70')
die()

im = PILImage.open(imgs[0])
display(im.resize((1100, int(1100*im.height/im.width))))


## 3. Full build (8-12 h)

Safe to re-run after a disconnect — every stage skips finished work.

Stage 1 prints the record count and split. **Sanity-check it before letting
stage 3 start:** you want ~37,225 records and a `STEMI+AF+LVH` row that is not zero.


In [ ]:
# Full build. run() now reads os.environ at call time, so ECG_IMAGE_BUILD is
# BUILD (set in section 0), not the smoke directory.
assert os.environ['ECG_IMAGE_BUILD'] == str(BUILD), os.environ['ECG_IMAGE_BUILD']
t0 = time.time()
out1 = run('stage1_manifest_from_csv.py', '--input-csv', str(CSV))


In [ ]:
section('[gate] manifest sanity')
# Refuse to burn 8 hours on a manifest that lost the multi-label classes.
man = pd.read_csv(BUILD/'image_manifest.csv')
counts = man.cls.value_counts()
print(counts.to_string()); print(f'\ntotal {len(man):,}')
check(f'record count {len(man):,} >= 35,000', len(man) >= 35000)
check(f"STEMI+AF+LVH present ({counts.get('STEMI+AF+LVH',0)})", counts.get('STEMI+AF+LVH',0) > 0)
check(f"AF+LVH >= 2,000 ({counts.get('AF+LVH',0)})", counts.get('AF+LVH',0) >= 2000)
die()


In [ ]:
section('[gate] stage 2')
out2 = run('stage2_transcode.py')
check('band printed as 0.05-150 Hz', '0.05-150 Hz' in out2)
check('no records lost to missing_metadata', 'missing_metadata' not in out2)
die()


In [ ]:
# The long one, ~5-8 h. Safe to re-run after a disconnect.
out3 = run('stage3_render.py', '-j', os.environ['ECG_WORKERS'], quiet=True)
print(f'elapsed so far: {(time.time()-t0)/3600:.1f} h')


In [ ]:
# ~2-4 h. Safe to re-run.
out4 = run('stage4_augment.py', '-j', os.environ['ECG_WORKERS'], quiet=True)
print(f'TOTAL elapsed: {(time.time()-t0)/3600:.1f} h')


In [ ]:
out5 = run('stage5_verify.py')


## 4. Package to Drive

**Never write 129k loose files to Drive.** Each one is a FUSE round-trip and you
will hit the API rate limit long before it finishes. Tar into a few large
archives instead: a handful of calls rather than a quarter of a million.


In [ ]:
idx = pd.read_csv(BUILD/'index.csv')
print(f'{len(idx):,} images')
tot = sum(f.stat().st_size for f in (BUILD/'images').rglob('*') if f.is_file())
print(f'{tot/2**30:.1f} GB to upload')
free_drive = None  # check your Google One quota; 117.8 GB needs the 200 GB plan

EXPDIR.mkdir(parents=True, exist_ok=True); RUNDIR.mkdir(parents=True, exist_ok=True)
# small artifacts first, so a later failure still leaves the manifest on Drive
idx.to_csv(RUNDIR/'index.csv', index=False)
(RUNDIR/'settings.json').write_text(json.dumps({
    'bandpass_hz': list(C.BANDPASS), 'dpi': C.DPI, 'print_header': C.PRINT_HEADER,
    'annotations': C.ANNOTATIONS, 'row_gap': C.ROW_GAP, 'rhythm_gap': C.RHYTHM_GAP,
    'variants': C.VARIANTS_PER_SPLIT, 'records': int(len(man)), 'images': int(len(idx)),
    'hours': round((time.time()-t0)/3600, 2)}, indent=2))
for name, txt in (('stage1.log',out1),('stage2.log',out2),('stage5.log',out5)):
    (RUNDIR/name).write_text(txt)
print('wrote index + settings + logs to', RUNDIR)


In [ ]:
# One archive per split. test/val are small; train is the bulk.
for split in ('test','val','train'):
    src = BUILD/'images'/split
    if not src.is_dir():
        continue
    dst = EXPDIR/f'images_{split}.tar'
    if dst.exists():
        print('skip (exists):', dst.name)
        continue
    print(f'packing {split} ...', flush=True)
    r = subprocess.run(['tar','-cf',str(dst),'-C',str(src),'.'],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1000:]); raise SystemExit(f'tar failed for {split}')
    print(f'  {dst.name}  {dst.stat().st_size/2**30:.1f} GB')
for t in sorted(EXPDIR.glob('images_*.tar')):
    print(f'  {t.name:24s} {t.stat().st_size/2**30:7.1f} GB')


In [ ]:
# Checksums, so a truncated upload is detectable later.
import hashlib
for t in sorted(EXPDIR.glob('images_*.tar')):
    s = EXPDIR/(t.name + '.sha256')
    if s.exists():
        continue
    h = hashlib.sha256()
    with open(t,'rb') as fh:
        for chunk in iter(lambda: fh.read(1<<24), b''):
            h.update(chunk)
    s.write_text(f'{h.hexdigest()}  {t.name}\n')
    print(s.name, 'written')
print('\nDONE.')
print('  corpus archives ->', EXPDIR)
print('  run metadata    ->', RUNDIR)
